# Task 1V - Sentiment Analysis on Twitter Data

**Objective:** Perform sentiment analysis on tweets using Python.  
Tweepy is used for Twitter/X API access, TextBlob is used for sentiment analysis, and Matplotlib is used for visualization.

### Libraries Used
- **Tweepy** – to access Twitter/X API and retrieve tweets
- **TextBlob** – to calculate sentiment polarity
- **Pandas** – to store and analyze tweet data
- **Matplotlib** – to visualize sentiment trends

In [ ]:
# Install the required libraries if needed
# !pip install tweepy textblob pandas matplotlib

import os
import tweepy
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob

## 1. Get Tweets using Tweepy

For API access, store your Twitter/X API Bearer Token in an environment variable named `TWITTER_BEARER_TOKEN`.

The code below searches for recent tweets containing a topic such as **AI**, **technology**, or any other keyword.

In [ ]:
# Add your Twitter/X Bearer Token as an environment variable before running this cell.
# Example in Windows:
# set TWITTER_BEARER_TOKEN=your_token_here

bearer_token = os.getenv("TWITTER_BEARER_TOKEN")

if bearer_token:
    client = tweepy.Client(bearer_token=bearer_token)

    query = "AI -is:retweet lang:en"
    response = client.search_recent_tweets(
        query=query,
        max_results=50,
        tweet_fields=["created_at", "text"]
    )

    tweets = []
    if response.data:
        for tweet in response.data:
            tweets.append({
                "date": tweet.created_at,
                "tweet": tweet.text
            })

    df = pd.DataFrame(tweets)
else:
    print("Twitter/X Bearer Token not found.")
    print("Set TWITTER_BEARER_TOKEN and run this cell again.")

## 2. Load Tweet Dataset

If API access is not available, a CSV file containing tweets can also be used.  
The CSV should contain at least a `tweet` column and preferably a `date` column.

In [ ]:
# If you are using a CSV instead of the API, uncomment and edit this line:
# df = pd.read_csv("tweets.csv")

if 'df' in globals() and not df.empty:
    print("Number of tweets:", len(df))
    display(df.head())
else:
    print("No tweet data loaded yet.")

## 3. Sentiment Analysis using TextBlob

TextBlob returns a **polarity** value from -1 to 1:
- Positive polarity -> Positive sentiment
- Negative polarity -> Negative sentiment
- Zero polarity -> Neutral sentiment

In [ ]:
def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        sentiment = "Positive"
    elif polarity < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return pd.Series([polarity, sentiment])

if 'df' in globals() and not df.empty:
    df[["polarity", "sentiment"]] = df["tweet"].apply(get_sentiment)
    display(df.head())
else:
    print("Load tweets first.")

## 4. Overall Sentiment

In [ ]:
if 'df' in globals() and not df.empty and 'sentiment' in df.columns:
    sentiment_counts = df["sentiment"].value_counts()
    print(sentiment_counts)

    sentiment_counts.plot(kind="bar", figsize=(7, 4))
    plt.title("Overall Tweet Sentiment")
    plt.xlabel("Sentiment")
    plt.ylabel("Number of Tweets")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 5. Sentiment Trend Over Time

In [ ]:
if 'df' in globals() and not df.empty and 'date' in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    daily_sentiment = (
        df.groupby([df["date"].dt.date, "sentiment"])
        .size()
        .unstack(fill_value=0)
    )

    daily_sentiment.plot(kind="line", marker="o", figsize=(10, 5))
    plt.title("Sentiment Trend Over Time")
    plt.xlabel("Date")
    plt.ylabel("Number of Tweets")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("A valid 'date' column is required for the time trend.")

## 6. Sentiment Percentages

In [ ]:
if 'df' in globals() and not df.empty and 'sentiment' in df.columns:
    percentages = df["sentiment"].value_counts(normalize=True) * 100
    print(percentages.round(2).astype(str) + "%")

## Observations

- TextBlob assigns a polarity score to each tweet.
- Tweets are grouped into positive, negative, and neutral categories.
- The bar chart shows the overall distribution of sentiments.
- The time-series chart helps identify changes in sentiment over time.
- The results depend on the selected tweets and the topic used for collection.

## Conclusion

Sentiment analysis can be used to understand the general opinion expressed in social media posts. In this task, Tweepy provides tweet data, TextBlob classifies the sentiment, and Matplotlib helps visualize the results. The same approach can be applied to different topics by changing the search query.